In [ ]:
!pip check

In [ ]:
!pip uninstall -y \
langchain-google-genai \
google-generativeai \
google-ai-generativelanguage

In [ ]:
!pip install \
langchain \
langchain-community \
langchain-huggingface \
faiss-cpu \
sentence-transformers \
pypdf \
transformers \
accelerate

In [ ]:
!pip install -q --force-reinstall requests==2.32.4

In [ ]:
!pip check

In [ ]:
import requests
print(requests.__version__)

In [ ]:
!pip check

In [ ]:
!pip freeze | grep requests
!pip check


In [ ]:
!pip install jedi

In [ ]:
!pip install -q \
requests==2.32.5 \
langchain==0.3.27 \
langchain-community==0.3.31 \
langchain-huggingface==0.3.1 \
faiss-cpu==1.12.0 \
sentence-transformers==5.1.0 \
transformers==4.55.4 \
unstructured==0.18.14

In [ ]:
!pip uninstall -y \
langchain-google-genai \
langgraph \
langgraph-prebuilt \
langchain-classic

In [ ]:
!pip install -q \
langchain==0.3.27 \
langchain-core==0.3.86 \
langchain-community==0.3.31 \
langchain-text-splitters==0.3.11 \
langchain-huggingface==0.3.1 \
faiss-cpu \
sentence-transformers

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.prompts import PromptTemplate

from langchain.chains import RetrievalQA

#import google.generativeai as genai

import pandas as pd
import numpy as np

from datetime import datetime

In [ ]:
!pip install -q langchain-google-genai==2.1.5

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

print("Import successful")

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
documents = []

for file_name in uploaded.keys():

    loader = PyPDFLoader(file_name)

    docs = loader.load()

    documents.extend(docs)

print("Total Pages:", len(documents))

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n","\n"," ",""]
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vector_db = FAISS.from_documents(
    chunks,
    embedding_model
)

vector_db.save_local("teaching_assistant_db")

print("FAISS Index Created")

In [ ]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k":5
    }
)

In [ ]:
import os

print("GOOGLE_API_KEY =", os.getenv("GOOGLE_API_KEY"))

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

In [ ]:
prompt_template = """
You are an AI Teaching Assistant.

Use ONLY the provided context.

If answer is not available in context,
say:
'I could not find this in the uploaded material.'

Context:
{context}

Question:
{question}

Provide:

1. Detailed explanation
2. Important points
3. Example if possible
4. Summary
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context","question"]
)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)


In [ ]:
question = input("Ask Question: ")

response = qa_chain.invoke(
    {"query":question}
)

print(response["result"])

In [ ]:
question = input("Ask Question: ")

response = qa_chain.invoke(
    {"query":question}
)

print(response["result"])

In [ ]:
question = input("Ask Question: ")

response = qa_chain.invoke(
    {"query":question}
)

print("\nANSWER\n")
print(response["result"])

print("\nSOURCES\n")

for doc in response["source_documents"]:

    print(
        doc.metadata.get("source"),
        "Page:",
        doc.metadata.get("page")
    )

In [ ]:
chat_history = []

In [ ]:
while True:

    q = input("\nYou: ")

    if q.lower()=="exit":
        break

    response = qa_chain.invoke(
        {"query":q}
    )

    answer = response["result"]

    chat_history.append({
        "question":q,
        "answer":answer,
        "time":str(datetime.now())
    })

    print("\nAssistant:\n")
    print(answer)

In [ ]:
history_df = pd.DataFrame(chat_history)

history_df.to_csv(
    "chat_history.csv",
    index=False
)

print("Saved")


In [ ]:
def generate_quiz(topic):

    prompt = f"""
    Generate 10 MCQs.

    Topic:
    {topic}

    Format:

    Question

    A.
    B.
    C.
    D.

    Correct Answer
    """

    response = llm.invoke(prompt)

    print(response.content)

In [ ]:
generate_quiz(
    "Machine Learning"
)

In [ ]:
def ask_with_level(question, level):

    prompt = f"""
    Explain in {level} level.

    Question:
    {question}
    """

    response = llm.invoke(prompt)

    return response.content

In [ ]:
ask_with_level(
    "What is PCA?",
    "Beginner"
)

In [ ]:
!pip install rank-bm25

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
texts = [doc.page_content for doc in chunks]

tokenized = [
    t.split()
    for t in texts
]

bm25 = BM25Okapi(tokenized)

In [ ]:
query = "What is gradient descent?"

scores = bm25.get_scores(
    query.split()
)

top_docs = np.argsort(scores)[::-1][:5]

for idx in top_docs:
    print(texts[idx][:500])

In [ ]:
import gradio as gr

In [ ]:
def chatbot(question):

    response = qa_chain.invoke(
        {"query":question}
    )

    return response["result"]

In [ ]:
app = gr.Interface(
    fn=chatbot,
    inputs="text",
    outputs="text",
    title="AI Teaching Assistant"
)

app.launch(
    share=True
)